# Praktikum 4 - Perbandingan Skalabilitas Metode ANN pada Skala Besar

**Mata Kuliah:** Pembelajaran Mesin
**Jobsheet 06:** ANN (Approximate Nearest Neighbors)

### Tujuan Praktikum:
Mengukur waktu komputasi pembangunan indeks (*build time*) dan latensi pencarian (*query time*) pada dataset berdimensi tinggi dengan 100.000 titik data.

## Langkah 1 - Pembuatan Dataset Sintetis Skala Besar (100.000 Vektor 5D)

In [1]:
import numpy as np
import time
import faiss

np.random.seed(42)
n_data = 100000
dim = 5
X = np.random.rand(n_data, dim).astype('float32')
query = np.random.rand(1, dim).astype('float32')
k = 5
print(f'Dataset uji: {n_data} sampel, dimensi = {dim}')

Dataset uji: 100000 sampel, dimensi = 5


## Langkah 2 - Benchmark FAISS (IndexFlatL2 dan IndexIVFFlat)

In [2]:
# 1. FAISS Flat (Exact L2)
start = time.time()
idx_flat = faiss.IndexFlatL2(dim)
idx_flat.add(X)
t_flat_build = time.time() - start

start = time.time()
dist_flat, idxs_flat = idx_flat.search(query, k)
t_flat_search = time.time() - start

# 2. FAISS IVF (Inverted File Index)
start = time.time()
quantizer = faiss.IndexFlatL2(dim)
nlist = 50
idx_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
idx_ivf.train(X)
idx_ivf.add(X)
t_ivf_build = time.time() - start

start = time.time()
idx_ivf.nprobe = 5
dist_ivf, idxs_ivf = idx_ivf.search(query, k)
t_ivf_search = time.time() - start

print(f'FAISS Flat -> Build: {t_flat_build*1000:.2f} ms | Search: {t_flat_search*1000:.3f} ms')
print(f'FAISS IVF  -> Build: {t_ivf_build*1000:.2f} ms | Search: {t_ivf_search*1000:.3f} ms')

FAISS Flat -> Build: 1.05 ms | Search: 3.488 ms
FAISS IVF  -> Build: 41.40 ms | Search: 1.949 ms
